# 📱 RubikCare Mobile - دليل BlazorWebView

**الإصدار: 2.0** | **التحديث: 2 مايو 2026**

---

## 📅 سجل التحديثات

| التاريخ | الإصدار | التغيير |
|----------|---------|----------|
| 11 أبريل 2026 | 1.0 | النسخة الأولى |
| **2 مايو 2026** | **2.0** | **أنماط BlazorWebView الناجحة، نمط SessionService، التنقل بين Blazor/XAML، حلول المشكلات الحقيقية** |

---

## 🧭 جدول المحتويات

1. [الأنماط الناجحة لـ BlazorWebView](#1-الأنماط-الناجحة)
2. [النمط الصحيح لإنشاء حاوية BlazorWebView](#2-النمط-الصحيح-للحاوية)
3. [تمرير البيانات بين XAML و Blazor](#3-تمرير-البيانات)
4. [التنقل من Blazor إلى صفحة XAML](#4-التنقل-من-blazor-إلى-xaml)
5. [التنقل من XAML إلى Blazor](#5-التنقل-من-xaml-إلى-blazor)
6. [مشكلة تعدد المستخدمين - SessionService](#6-مشكلة-تعدد-المستخدمين)
7. [مشكلة SecureStorage في Shared.UI](#7-مشكلة-securestorage)
8. [مشكلة Reload البيضاء](#8-مشكلة-reload-البيضاء)
9. [الأنماط التي يجب تجنبها](#9-أنماط-فاشلة-لا-تكررها)
10. [قائمة التحقق](#10-قائمة-التحقق)

---

## 1. الأنماط الناجحة

### ✅ النمط الذهبي: حاوية XAML منفصلة لكل صفحة

```
Mobile/Features/Shared/Views/
├── MyPageFlow.xaml       ← حاوية XAML
├── MyPageFlow.xaml.cs    ← كود خلفي
```

**مبدأ:** صفحة Blazor واحدة = حاوية XAML واحدة.

---

## 2. النمط الصحيح للحاوية

### `MyPageFlow.xaml`:

```xml
<?xml version="1.0" encoding="utf-8" ?>
<ContentPage xmlns="http://schemas.microsoft.com/dotnet/2021/maui"
             xmlns:x="http://schemas.microsoft.com/winfx/2009/xaml"
             xmlns:webview="clr-namespace:Microsoft.AspNetCore.Components.WebView.Maui;assembly=Microsoft.AspNetCore.Components.WebView.Maui"
             x:Class="RubikCare.Mobile.Features.Shared.Views.MyPageFlow"
             Title="عنوان الصفحة">

    <webview:BlazorWebView x:Name="blazorWebView" HostPage="wwwroot/index.html" />

</ContentPage>
```

### `MyPageFlow.xaml.cs`:

```csharp
using Microsoft.AspNetCore.Components.WebView.Maui;
using RubikCare.Shared.UI.Components.MyFeature;

namespace RubikCare.Mobile.Features.Shared.Views;

public partial class MyPageFlow : ContentPage
{
    public MyPageFlow()
    {
        InitializeComponent();

        // ⭐ RootComponent في الـ constructor - وليس في OnNavigatedTo
        blazorWebView.RootComponents.Add(new RootComponent
        {
            Selector = "#app",
            ComponentType = typeof(MyPageComponent),
            Parameters = new Dictionary<string, object?>
            {
                { "ParameterName", parameterValue }
            }
        });
    }
}
```

### ⚠️ لا تفعل هذا:

```csharp
// ❌ خطأ - RootComponent في OnNavigatedTo
protected override void OnNavigatedTo(NavigatedToEventArgs args)
{
    blazorWebView.RootComponents.Add(new RootComponent { ... });  // يسبب reload أو تكرار
}
```

---

## 3. تمرير البيانات بين XAML و Blazor

### ✅ النمط الصحيح: `Action<T>` بدلاً من `EventCallback`

**من XAML إلى Blazor (معاملات):**

```csharp
// في الحاوية XAML
blazorWebView.RootComponents.Add(new RootComponent
{
    Selector = "#app",
    ComponentType = typeof(MyPageComponent),
    Parameters = new Dictionary<string, object?>
    {
        { "PatientId", 15 },
        { "OnItemSelected", (Action<MyModel>)OnItemSelected }
    }
});
```

**في صفحة Blazor:**

```csharp
// في MyPageComponent.razor.cs
[Parameter] public int PatientId { get; set; }
[Parameter] public Action<MyModel>? OnItemSelected { get; set; }

private void SelectItem(MyModel item)
{
    OnItemSelected?.Invoke(item);
}
```

### ❌ لا تستخدم `EventCallback` عبر `RootComponent.Parameters`:

```csharp
// ❌ لا يعمل بشكل موثوق
{ "OnItemSelected", EventCallback.Factory.Create<MyModel>(this, OnItemSelected) }
```

---

## 4. التنقل من Blazor إلى XAML

### ✅ النمط الصحيح: `Action` + `Shell.GoToAsync`

**في الحاوية XAML:**

```csharp
private async void OnItemSelected(MyModel item)
{
    await Shell.Current.GoToAsync("///TargetXamlPage", new Dictionary<string, object>
    {
        { "parameterName", item.Id.ToString() }
    });
}
```

**في Blazor (Fallback للويب):**

```csharp
private void SelectItem(MyModel item)
{
    if (OnItemSelected != null)
        OnItemSelected.Invoke(item);
    else
        Navigation.NavigateTo($"/web-page/{item.Id}");
}
```

---

## 5. التنقل من XAML إلى Blazor

### ✅ النمط الصحيح: `Shell.GoToAsync` للحاوية

```csharp
// في أي ViewModel أو صفحة XAML
await Shell.Current.GoToAsync("///support");  // يفتح SupportFlow.xaml (اللي تحوي BlazorWebView)
```

---

## 6. مشكلة تعدد المستخدمين - `IPatientSessionService`

### المشكلة:

`BlazorWebView` يحتفظ بحالة الـ Component بين التنقلات. `OnInitializedAsync` لا تُستدعى مرة أخرى عند تغيير المستخدم. `Parameters` الجديدة لا تُشغّل إعادة تهيئة.

### ✅ الحل: Singleton Service + Event

**الملف:** `Shared.UI/Services/ISessionService.cs`

```csharp
namespace RubikCare.Shared.UI.Services;

public interface IPatientSessionService
{
    int PatientId { get; }
    event Action OnSessionChanged;
    void UpdateSession(int patientId);
    void ClearSession();
}

public class PatientSessionService : IPatientSessionService
{
    private int _patientId;
    public int PatientId => _patientId;
    public event Action? OnSessionChanged;

    public void UpdateSession(int patientId)
    {
        _patientId = patientId;
        OnSessionChanged?.Invoke();
    }

    public void ClearSession()
    {
        _patientId = 0;
        OnSessionChanged?.Invoke();
    }
}
```

**التسجيل في `MauiProgram.cs`:**

```csharp
builder.Services.AddSingleton<IPatientSessionService, PatientSessionService>();
```

**في الحاوية XAML:**

```csharp
public partial class MyPageFlow : ContentPage
{
    private readonly IPatientSessionService _sessionService;

    public MyPageFlow(IPatientSessionService sessionService)
    {
        InitializeComponent();
        _sessionService = sessionService;

        blazorWebView.RootComponents.Add(new RootComponent
        {
            Selector = "#app",
            ComponentType = typeof(MyPageComponent),
            Parameters = new Dictionary<string, object?>
            {
                { "OnItemSelected", (Action<MyModel>)OnItemSelected }
            }
        });
    }

    protected override async void OnNavigatedTo(NavigatedToEventArgs args)
    {
        base.OnNavigatedTo(args);
        var patientId = await GetPatientIdFromSecureStorage();
        _sessionService.UpdateSession(patientId);  // ⭐ يخطر Blazor
    }
}
```

**في صفحة Blazor:**

```csharp
public partial class MyPageComponent : ComponentBase, IDisposable
{
    [Inject] protected IPatientSessionService SessionService { get; set; } = default!;

    protected override async Task OnInitializedAsync()
    {
        SessionService.OnSessionChanged += OnSessionChangedHandler;
        await LoadData(SessionService.PatientId);
    }

    private async void OnSessionChangedHandler()
    {
        await InvokeAsync(async () => await LoadData(SessionService.PatientId));
    }

    public void Dispose()
    {
        SessionService.OnSessionChanged -= OnSessionChangedHandler;
    }
}
```

---

## 7. مشكلة SecureStorage في Shared.UI

### ❌ لا تفعل هذا:

```csharp
// Shared.UI لا يرى MAUI SecureStorage!
[Inject] protected ISecureStorage SecureStorage { get; set; }
```

### ✅ الحل: MAUI يقرأ SecureStorage ويمرر لـ Blazor

```csharp
// في الحاوية XAML
var patientId = await SecureStorage.GetAsync("user_profile_id");
_sessionService.UpdateSession(patientId);  // Blazor يقرأ من SessionService
```

---

## 8. مشكلة Reload البيضاء

### الأسباب والحلول:

| السبب | الحل |
|-------|------|
| `RootComponent` في `OnNavigatedTo` | انقله لـ constructor |
| `HttpClient` غير مسجل في `MauiProgram` | سجله أو استخدم `ApiService` |
| `SecureStorage` داخل Blazor | استخدم `SessionService` بدلاً منه |
| `@inject` لخدمة غير موجودة | تأكد من التسجيل في `MauiProgram.cs` |

---

## 9. أنماط فاشلة (لا تكررها)

| # | النمط الفاشل | لماذا فشل | البديل الصحيح |
|---|-------------|-----------|----------------|
| 1 | `HttpClient` مباشر في Shared.UI | غير مسجل في Mobile | `ApiService` أو `SessionService` |
| 2 | `SecureStorage` في Blazor | Shared.UI لا يرى MAUI | تمرير من XAML |
| 3 | `EventCallback` عبر `RootComponent.Parameters` | لا يعمل بشكل موثوق | `Action<T>` |
| 4 | `RootComponent` في `OnNavigatedTo` | يسبب reload أبيض | `RootComponent` في constructor |
| 5 | `Navigation.NavigateTo` للانتقال لـ XAML | يبحث عن صفحة Razor | `OnItemSelected` + `Shell.GoToAsync` |
| 6 | مشاركة `BlazorPage` الديناميكية | `QueryProperty` لا يعمل على Android | حاوية منفصلة لكل صفحة |
| 7 | `@inject` لخدمة مسجلة في Web فقط | غير موجودة في Mobile | Interface في Shared.UI + تنفيذ في Mobile |

---

## 10. قائمة التحقق

### ✅ قبل إنشاء صفحة BlazorWebView جديدة:

- [ ] هل أنشأت حاوية XAML منفصلة (`Flow.xaml` + `Flow.xaml.cs`)؟
- [ ] هل `RootComponent` في الـ constructor وليس في `OnNavigatedTo`؟
- [ ] هل استخدمت `Action<T>` بدلاً من `EventCallback` للتنقل؟
- [ ] هل `IPatientSessionService` هو مصدر `PatientId`؟
- [ ] هل تجنبت `SecureStorage` في Shared.UI؟
- [ ] هل سجلت الحاوية في `MauiProgram.cs` (`AddTransient`)?
- [ ] هل سجلت المسار في `AppShell.xaml` + `AppShell.xaml.cs`؟
- [ ] هل اختبرت على Windows قبل Android؟

### ✅ قبل رفع Commit:

- [ ] هل الصفحة تعمل على Windows Machine؟
- [ ] هل اختبرت مع مستخدمين مختلفين؟
- [ ] هل `Dispose()` موجود لإلغاء الاشتراك من الأحداث؟
- [ ] هل لا توجد `Console.WriteLine` للتشخيص؟

---

**تم التحديث في: 2 مايو 2026** | **الإصدار: 2.0**

## 11. إدارة بيئة النشر (Release Management)

### ⚠️ المشكلة الشائعة

`ApiService` يحتوي على `#if DEBUG` مما يجعل سلوك التطبيق مختلفاً بين Debug و Release.

### ✅ الحل: ملف إعدادات مستقل للموبايل

**1. أنشئ:** `Mobile/Resources/Raw/appsettings.json`

```json
{
  "ApiBaseUrl": "http://161.97.105.83:5235/"
}
```

**2. في `ApiService`، أضف دالة لقراءة الإعدادات:**

```csharp
private static string GetBaseUrlFromConfig()
{
    try
    {
        var assembly = Assembly.GetExecutingAssembly();
        using var stream = assembly.GetManifestResourceStream("RubikCare.Mobile.Resources.Raw.appsettings.json");
        if (stream != null)
        {
            using var reader = new StreamReader(stream);
            var json = reader.ReadToEnd();
            var config = JsonSerializer.Deserialize<AppConfig>(json);
            return config?.ApiBaseUrl ?? "http://localhost:5235/";
        }
    }
    catch { }
    return "http://localhost:5235/";
}

public class AppConfig
{
    public string ApiBaseUrl { get; set; } = "http://localhost:5235/";
}
```

**3. في `.csproj`، تأكد من تضمين الملف:**

```xml
<ItemGroup>
    <EmbeddedResource Include="Resources\Raw\appsettings.json" />
</ItemGroup>
```

**4. استخدمها في `GetBaseUrl`:**

```csharp
private string GetBaseUrl()
{
    return GetBaseUrlFromConfig();
}
```

---

### 📊 النتيجة

| البيئة | طريقة التغيير |
|--------|---------------|
| **تطوير محلي** | عدل `appsettings.json` ← `http://localhost:5235/` |
| **اختبار على Android** | عدل `appsettings.json` ← `http://192.168.1.41:5235/` |
| **إنتاج** | عدل `appsettings.json` ← `http://161.97.105.83:5235/` |

**بدون تغيير الكود. بدون `#if DEBUG`. ملف واحد.**

---

**تم تحديث الدليل في: 2 مايو 2026**


## 📘 **ملحق: الأنماط الأساسية لـ Blazor Hybrid في MAUI**

### **النمط 1: Bootstrap API + Cached Session Service**

**المشكلة:** كل صفحة تقوم بجلب بيانات المستخدم والبرامج والمؤسسات بشكل منفصل، مما يؤدي إلى تكرار الاستدعاءات وبطء الأداء.

**الحل:** API واحد لجلب كل البيانات عند بدء التشغيل + Service لتخزينها في الذاكرة المؤقتة.

**الهيكل:**

```
Mobile/
├── Infrastructure/Services/
│   └── CachedUserSessionService.cs
└── ViewModels/
    └── AppShellViewModel.cs (يستدعي الـ Bootstrap API مرة واحدة)

Api.Web/Controllers/
└── UserController.cs (POST api/user/session-bootstrap)
```

**مقتطفات من الكود:**

```csharp
// CachedUserSessionService.cs
public interface ICachedUserSessionService
{
    Task<UserSessionData?> GetSessionAsync();
    Task RefreshSessionAsync();
}

// AppShellViewModel.cs
public async Task LoadUserDataAsync()
{
    var session = await _cachedSessionService.GetSessionAsync();
    if (session != null)
    {
        UserName = session.FullNameAr;
        UserRole = session.Roles.FirstOrDefault();
        // ... تعبئة باقي الخصائص
    }
}
```

---

### **النمط 2: Single Source of Truth للـ DTOs**

**المشكلة:** تعريف نفس الـ DTO في `Application.DTOs` و `Shared.UI.DTOs` يؤدي إلى صراعات وأخطاء غير مفهومة.

**القاعدة الذهبية:**
-   `Application.DTOs` هو المصدر الوحيد والصحيح.
-   `Shared.UI` لا يعرّف أي DTO خاص به. يستخدم DTOs من `Application`.

**مثال:**

```csharp
// ✅ صحيح - استخدام DTO من Application
@using RubikCare.Application.DTOs.Messaging
@inject IApiService ApiService

private List<MessageResponseDto> messages = new();

// ❌ خطأ - تعريف DTO جديد في Shared.UI
public class MyMessageDto { ... }
```

---

### **النمط 3: تهيئة BlazorWebView الآمنة**

**المشكلة:** تعريف `RootComponent` في XAML لا يعمل بشكل موثوق، خاصة عند تمرير `Parameters`.

**الحل:** تعريف `RootComponent` في الكود الخلفي (Code-behind) فقط.

**الصحيح ✅:**

```xml
<!-- MyPageFlow.xaml -->
<webview:BlazorWebView x:Name="blazorWebView" HostPage="wwwroot/index.html" />
```

```csharp
// MyPageFlow.xaml.cs
public MyPageFlow()
{
    InitializeComponent();
    blazorWebView.RootComponents.Add(new RootComponent
    {
        Selector = "#app",
        ComponentType = typeof(MyPage),
        Parameters = new Dictionary<string, object?>
        {
            { "ParameterName", parameterValue }
        }
    });
}
```

**الخطأ ❌:**

```xml
<!-- لا تفعل هذا -->
<webview:BlazorWebView>
    <webview:BlazorWebView.RootComponents>
        <webview:RootComponent Selector="#app" ComponentType="..." />
    </webview:BlazorWebView.RootComponents>
</webview:BlazorWebView>
```

---

### **النمط 4: الوصول إلى سياق المستخدم (User Context)**

**المشكلة:** لا توجد طريقة موحدة لجلب `UserProfileId` في صفحات `Shared.UI`.

**الحل:** تمرير `Action<int>` من الحاوية إلى صفحة Blazor.

**في الحاوية XAML:**

```csharp
// DoctorSearchFlow.xaml.cs
blazorWebView.RootComponents.Add(new RootComponent
{
    Selector = "#app",
    ComponentType = typeof(DoctorSearchPage),
    Parameters = new Dictionary<string, object?>
    {
        { "OnSelectClinic", new Action<int>(async (clinicId) =>
            {
                await Shell.Current.GoToAsync($"doctorprofile?clinicId={clinicId}");
            })
        }
    }
});
```

**في صفحة Blazor:**

```csharp
// DoctorSearchPage.razor
[Parameter] public Action<int>? OnSelectClinic { get; set; }

private void SelectClinic(int clinicId)
{
    OnSelectClinic?.Invoke(clinicId);
}
```

---

### **النمط 5: Debugging في MAUI Blazor (الأخطاء الصامتة)**

**المشكلة:** بعض الأخطاء تؤدي إلى خروج التطبيق (Exit code 0) أو ظهور كلمة "Reload" فقط.

**خطوات التشخيص:**
1.  **إضافة `Debug.WriteLine` في كل دالة:**

```csharp
System.Diagnostics.Debug.WriteLine($"🟢 MethodName - Start, Parameter={parameter}");
```

2.  **مراقبة Output window في Visual Studio**، وليس Console التطبيق.

3.  **التحقق من `wwwroot/index.html`** - تأكد من وجود `<div id="app">Loading...</div>`.

4.  **تسجيل `RootComponent` في Code-behind فقط** (كما في النمط 3).

5.  **التأكد من تسجيل جميع الصفحات في `MauiProgram.cs`:**

```csharp
builder.Services.AddTransient<MyPageFlow>();
```

---

### **النمط 6: CSS في Blazor Hybrid**

**المشكلة:** ملفات CSS لا تُحمّل أو لا تعمل بالشكل المتوقع.

**الأسباب والحلول:**

| السبب | الحل |
|-------|------|
| ملف CSS في `wwwroot` | ✅ مسار الملف يتم التعامل معه تلقائياً |
| ملف `.razor.css` (Scoped) | ✅ يعمل بشكل طبيعي |
| استيراد CSS عبر `@import` | تأكد من المسار الصحيح في `main.css` |
| CSS لا يُطبق | جرب وضع الأنماط مباشرة في HTML للاختبار (`<div style="color:red">`)

**توصية:** استخدم Scoped CSS (`.razor.css`) لأنه يعمل تلقائياً في Blazor Hybrid.

```css
/* DoctorSearchPage.razor.css */
.doctor-search-container {
    padding: 16px;
    background: #f5f7fa;
}
```

---

## ✅ **قائمة مراجعة سريعة (Cheat Sheet)**

| # | السؤال | الجواب الصحيح |
|---|--------|----------------|
| 1 | أين تعرّف `RootComponent`؟ | في Code-behind فقط |
| 2 | كيف تمرر Parameters من XAML إلى Blazor؟ | `new Action<T>(...)` |
| 3 | كيف تنتقل من Blazor إلى XAML؟ | `Action` في Page + `Shell.GoToAsync` في Flow |
| 4 | أين تعرّف DTOs؟ | في `Application.DTOs` فقط |
| 5 | كيف تجلب بيانات المستخدم مرة واحدة؟ | Bootstrap API + Cached Session Service |
| 6 | كيف تشخص الأخطاء الصامتة؟ | `Debug.WriteLine` + Output window |
